#### Windows-safe Wrapper

In [ ]:
from pathlib import Path
import shutil
import time
import gc
import matplotlib.pyplot as plt

def prepare_spicysnow_workdir(work_dir):
    """
    Create work_dir, then clean/recreate work_dir/tmp in a Windows-friendly way.
    No changes to spicy-snow required.
    """
    work_dir = Path(work_dir)
    tmp = work_dir / "tmp"

    # Make sure base folder exists
    work_dir.mkdir(parents=True, exist_ok=True)

    # Best-effort release of common handles (plots, rasterio-backed xarray objects, etc.)
    plt.close("all")
    gc.collect()
    time.sleep(0.5)

    # Robust delete with a few retries (handles brief locks from AV/sync/indexing)
    for attempt in range(5):
        try:
            if tmp.exists():
                shutil.rmtree(tmp)
            break
        except PermissionError:
            time.sleep(1.0)
            gc.collect()
    else:
        raise PermissionError(
            f"Could not delete temp folder (still locked): {tmp}\n"
            "Close Explorer windows/QGIS/ArcGIS, then restart the kernel and retry."
        )

    tmp.mkdir(parents=True, exist_ok=True)
    return work_dir

# --- Use it like this ---
work_dir = prepare_spicysnow_workdir(r"C:\SpicySnow")  # shorter path is often best on Windows
s1_sd = retrieve_snow_depth(area, dates, work_dir=str(work_dir))
